# TeleQnA — Agentic RAG Reference Solution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/Sandbox_Training/blob/main/notebooks/teleqna-agentic-rag-reference/notebook/TeleQnA_Agentic_RAG_Reference.ipynb)

A **retrieval-augmented, agentic** reference for the telecom-knowledge benchmark **TeleQnA** (10,000 multiple-choice questions). It wires a small language model and a retriever into an agent with three tools — **`retrieve()`**, **`answer()`**, **`refuse()`** — and a confidence **guardrail**.

**Pipeline (Y.3172 view):** regulatory/standards text → chunk + embed (retriever) → small LLM answers → **P-node guardrail** (refuse when retrieval confidence is low) → cited answer.

This notebook runs a **real small LLM** (`google/flan-t5-base`) on CPU — it works on the Colab free tier, just give it a minute to download the model. The dataset zip is password-protected (password documented in the TeleQnA repo README) to avoid training-data contamination.

In [ ]:
!pip -q install pyzipper

import io, json, urllib.request, re, random
import numpy as np
import pyzipper

ZIP_URL = "https://raw.githubusercontent.com/netop-team/TeleQnA/main/TeleQnA.zip"
PASSWORD = b"teleqnadataset"  # documented in the TeleQnA README (anti-contamination)

urllib.request.urlretrieve(ZIP_URL, "TeleQnA.zip")
with pyzipper.AESZipFile("TeleQnA.zip") as z:
    z.setpassword(PASSWORD)
    data = json.loads(z.read("TeleQnA.txt"))
print("Loaded", len(data), "questions")

## 1. Parse questions and build pools

Each item has a question, up to 5 options, the correct answer (`option N: ...`), an explanation, and a category. We use a **retrieval pool** (knowledge base, from explanations) and a **disjoint evaluation set**, so the answer to an eval question is never in the retrieved context (no leakage).

In [ ]:
def options_of(v):
    return {int(k.split()[1]): v[k] for k in v if k.startswith('option ')}

def correct_idx(v):
    m = re.search(r'option\s*(\d+)', v.get('answer', ''))
    return int(m.group(1)) if m else None

items = [v for v in data.values() if correct_idx(v) is not None and options_of(v)]
random.Random(42).shuffle(items)
POOL = items[:500]       # knowledge base documents
EVAL = items[500:525]    # questions the agent must answer (disjoint from POOL)

kb_docs = [f"{v['question']} {v.get('explanation', '')}".strip() for v in POOL]
print("KB documents:", len(kb_docs), "| eval questions:", len(EVAL))

## 2. Load the retriever and the LLM

`all-MiniLM-L6-v2` embeds the knowledge base for cosine-similarity retrieval; `flan-t5-base` is the answering LLM. Both download on first use (a minute or two on free tier).

In [ ]:
!pip -q install sentence-transformers transformers

from sentence_transformers import SentenceTransformer
from transformers import pipeline

embedder = SentenceTransformer("all-MiniLM-L6-v2")
kb_emb = embedder.encode(kb_docs, normalize_embeddings=True, show_progress_bar=False)

llm = pipeline("text2text-generation", model="google/flan-t5-base")
print("Retriever + LLM ready. KB embedding shape:", kb_emb.shape)

## 3. The agent and its tools

- **`retrieve(query, k)`** — embed the query, return the top-k KB docs and the best similarity (the confidence signal).
- **`answer(question, options, context)`** — prompt the LLM with the retrieved context and parse the chosen option number.
- **`refuse()`** — the **guardrail**: if retrieval confidence is below `TAU`, decline instead of guessing (maps to the UAE AI Charter's human-oversight / robustness principles).

Every tool call is logged, so the agent's behaviour is auditable.

In [ ]:
TAU = 0.25  # refuse below this retrieval confidence

def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    sims = kb_emb @ q
    top = np.argsort(sims)[::-1][:k]
    return [kb_docs[i] for i in top], float(sims[top[0]])

def answer(question, options, context):
    opts = "\n".join(f"{n}) {t}" for n, t in options.items())
    prompt = ("Use the context to answer the telecom multiple-choice question.\n"
              f"Context:\n{context}\n\nQuestion: {question}\nOptions:\n{opts}\n"
              "Reply with only the number of the correct option.")
    out = llm(prompt, max_new_tokens=5)[0]["generated_text"]
    m = re.search(r"\d+", out)
    return int(m.group()) if m and int(m.group()) in options else None

def agent(v, log):
    q = v["question"]; options = options_of(v)
    docs, conf = retrieve(q + " " + " ".join(options.values()))
    log.append(("retrieve", round(conf, 3)))
    if conf < TAU:
        log.append(("refuse", None))
        return None
    pred = answer(q, options, "\n".join(docs))
    log.append(("answer", pred))
    return pred

## 4. Run the agent over the evaluation set

In [ ]:
correct = answered = refused = 0
for v in EVAL:
    log = []
    pred = agent(v, log)
    gold = correct_idx(v)
    if pred is None:
        refused += 1
    else:
        answered += 1
        correct += int(pred == gold)

acc = correct / answered if answered else 0.0
print(f"Answered: {answered}/{len(EVAL)}  |  Refused (guardrail): {refused}")
print(f"Accuracy on answered questions: {acc:.2f}")

## 5. Inspect one agent trace

In [ ]:
v = EVAL[0]
log = []
pred = agent(v, log)
print("Q:", v["question"])
for n, t in options_of(v).items():
    print(f"   {n}) {t}")
print("Tool calls:", log)
print("Agent answer:", pred, "| gold:", correct_idx(v))

## 6. Notes for the leadership session

- **Why agentic, not a single prompt:** the agent *chooses* to retrieve, then answer or **refuse**. The refusal guardrail is a governance control, not a model feature — it is where policy lives.
- **Guardrails map to the UAE AI Charter 2024** (human oversight, robustness, accountability, transparency): low-confidence retrieval → refuse; every tool call is logged for audit.
- **Scaling up:** replace `flan-t5-base` with a stronger instruct model, and point the retriever at real 3GPP/IEEE specifications instead of the sampled explanations. The *architecture* — retrieve → answer → guardrail → cite — stays the same.
- The small CPU model is a **floor, not a ceiling**: accuracy here illustrates the pipeline; production systems add reranking, citations and stronger models.